# Fintech Privacy Filter — Training Notebook

Fine-tunes `OpenMed/privacy-filter-nemotron` on the `msdakot/fintech-privacy-pii` dataset.

**Runtime:** ~2–4 hours on T4

> **Keep your browser open for the full run.** Colab Pro keeps the runtime alive as long as the browser is running — the tab can be in the background but don't close the browser or sleep your laptop. `opf train` writes the checkpoint at the end, not incrementally, so a mid-run disconnect loses progress. If you need background execution, upgrade to Colab Pro+.

---
### Checklist before running
- [ ] Runtime → Change runtime type → **T4 GPU**
- [ ] Left sidebar → Key icon → Add secret `HF_TOKEN` (HuggingFace write-access token)
- [ ] Left sidebar → Key icon → Add secret `WANDB_API_KEY` (from wandb.ai/authorize)
- [ ] ~5 GB free on Google Drive
- [ ] Run cells top to bottom

## Preparation

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import wandb
from datasets import load_dataset
from google.colab import drive, userdata
from huggingface_hub import HfApi, create_repo, snapshot_download

In [ ]:
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch runtime to T4 before continuing.')

In [ ]:
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/fintech-pii-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

In [ ]:
!pip install -q 'opf @ git+https://github.com/openai/privacy-filter.git' datasets huggingface_hub wandb
print('Dependencies installed.')

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
user = HfApi(token=HF_TOKEN).whoami()
print(f"HF authenticated as: {user['name']}")

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
wandb.login()
print('W&B authenticated.')

## Data & Model

In [ ]:
BASE_MODEL_DIR = '/content/privacy-filter-base'

if not os.path.exists(BASE_MODEL_DIR):
    print('Downloading openai/privacy-filter...')
    snapshot_download(
        repo_id='openai/privacy-filter',
        local_dir=BASE_MODEL_DIR,
        token=HF_TOKEN,
    )
    print('Download complete.')
else:
    print('Base model already cached.')

print('Checkpoint files:', os.listdir(BASE_MODEL_DIR))

In [ ]:
DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

def record_to_opf(record):
    """Convert HF dataset record → opf JSONL format.
    HF:  spans = [{start, end, label}, ...]
    opf: spans = {label: [[start, end], ...], ...}
    """
    spans_dict = {}
    for span in record.get('spans', []):
        label = span['label']
        spans_dict.setdefault(label, []).append([span['start'], span['end']])
    return {'text': record['text'], 'spans': spans_dict}

print('Loading msdakot/fintech-privacy-pii from HF Hub...')
ds = load_dataset('msdakot/fintech-privacy-pii', token=HF_TOKEN)

split_sizes = {}
for split in ('train', 'validation', 'test'):
    path = DATA_DIR / f'{split}.jsonl'
    with open(path, 'w', encoding='utf-8') as f:
        for record in ds[split]:
            f.write(json.dumps(record_to_opf(record), ensure_ascii=False) + '\n')
    split_sizes[split] = len(ds[split])
    print(f'  {split}: {len(ds[split])} records → {path}')

print('Export complete.')

In [ ]:
label_space = {
    'span_class_names': [
        'O',
        'account_number', 'age', 'api_key', 'bank_routing_number',
        'biometric_identifier', 'blood_type', 'certificate_license_number',
        'city', 'company_name', 'coordinate', 'country', 'county',
        'credit_debit_card', 'customer_id', 'cvv', 'date', 'date_of_birth',
        'date_time', 'device_identifier', 'education_level', 'email',
        'employee_id', 'employment_status', 'fax_number', 'first_name',
        'gender', 'health_plan_beneficiary_number', 'http_cookie',
        'ipv4', 'ipv6', 'language', 'last_name', 'license_plate',
        'mac_address', 'medical_record_number', 'national_id', 'occupation',
        'password', 'phone_number', 'pin', 'political_view', 'postcode',
        'race_ethnicity', 'religious_belief', 'sexuality', 'ssn', 'state',
        'street_address', 'swift_bic', 'tax_id', 'time', 'unique_id',
        'url', 'user_name', 'vehicle_identifier',
        'iban', 'bban', 'lei', 'isin', 'cusip',
        'swift_mt_ref', 'policy_number', 'vat_number', 'loan_number', 'crypto_address',
    ]
}

LABEL_SPACE_PATH = '/content/label_space.json'
with open(LABEL_SPACE_PATH, 'w') as f:
    json.dump(label_space, f, indent=2)

print(f'Label space: {len(label_space["span_class_names"]) - 1} entity types + O')

## Sanity Check

1 epoch on 200 examples — verifies data format, checkpoint, and label space before the full run. Fix errors here, not mid-training.

In [ ]:
SANITY_DIR = '/content/sanity-check'
os.makedirs(SANITY_DIR, exist_ok=True)

sanity_cmd = [
    'opf', 'train', str(DATA_DIR / 'train.jsonl'),
    '--validation-dataset', str(DATA_DIR / 'validation.jsonl'),
    '--checkpoint', BASE_MODEL_DIR,
    '--label-space-json', LABEL_SPACE_PATH,
    '--output-dir', SANITY_DIR,
    '--overwrite-output',
    '--max-train-examples', '200',
    '--max-validation-examples', '50',
    '--epochs', '1',
    '--batch-size', '2',
    '--grad-accum-steps', '4',
    '--learning-rate', '1e-4',
    '--weight-decay', '0.0',
    '--output-param-dtype', 'bf16',
]

print('Running sanity check...')
result = subprocess.run(sanity_cmd)

if result.returncode != 0:
    raise RuntimeError('Sanity check failed — fix the error above before running full training.')

print('\nSanity check passed. Checkpoint files:', os.listdir(SANITY_DIR))
print('Proceed to full training.')

## Training

Initialises W&B then runs the full 3-epoch job (~2–4 hrs on T4). Keep the browser open — checkpoint is written at the end of training.

In [ ]:
run = wandb.init(
    project='fintech-privacy-filter',
    name='opf-base-fintech-v0',
    config={
        'base_model': 'openai/privacy-filter',
        'output_model': 'msdakot/fintech-privacy-filter-v0',
        'dataset': 'msdakot/fintech-privacy-pii',
        'train_examples': split_sizes['train'],
        'val_examples': split_sizes['validation'],
        'test_examples': split_sizes['test'],
        'languages': ['en', 'es', 'de', 'fr', 'it', 'nl', 'sv'],
        'n_entity_types': len(label_space['span_class_names']) - 1,
        'new_fintech_labels': 10,
        'learning_rate': 1e-4,
        'epochs': 3,
        'batch_size': 2,
        'grad_accum_steps': 4,
        'effective_batch_size': 8,
        'weight_decay': 0.0,
        'output_param_dtype': 'bf16',
        'hardware': 'T4',
    }
)

print(f'W&B run: {run.url}')

In [ ]:
cmd = [
    'opf', 'train', str(DATA_DIR / 'train.jsonl'),
    '--validation-dataset', str(DATA_DIR / 'validation.jsonl'),
    '--checkpoint', BASE_MODEL_DIR,
    '--label-space-json', LABEL_SPACE_PATH,
    '--output-dir', CHECKPOINT_DIR,
    '--overwrite-output',
    '--epochs', '3',
    '--batch-size', '2',
    '--grad-accum-steps', '4',
    '--learning-rate', '1e-4',
    '--weight-decay', '0.0',
    '--output-param-dtype', 'bf16',
]

print('Command:', ' '.join(cmd))
print('Training started...')
print('-' * 60)

t0 = time.time()
result = subprocess.run(cmd)
elapsed = time.time() - t0

wandb.log({'training_duration_minutes': elapsed / 60})

if result.returncode != 0:
    wandb.finish(exit_code=result.returncode)
    raise RuntimeError(f'opf train failed with code {result.returncode}')

print(f'\nTraining complete in {elapsed/60:.1f} minutes.')

## Results

In [ ]:
print('Checkpoint contents:', os.listdir(CHECKPOINT_DIR))

summary_path = os.path.join(CHECKPOINT_DIR, 'finetune_summary.json')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print('\nTraining summary:')
    print(json.dumps(summary, indent=2))

    def _flatten(d, prefix=''):
        out = {}
        for k, v in d.items():
            key = f'{prefix}/{k}' if prefix else k
            if isinstance(v, dict):
                out.update(_flatten(v, key))
            elif isinstance(v, (int, float)):
                out[key] = v
        return out

    wandb.log(_flatten(summary, 'train'))
    print('\nSummary logged to W&B.')
else:
    print('WARNING: finetune_summary.json not found.')

In [ ]:
HF_MODEL_REPO = 'msdakot/fintech-privacy-filter-v0'
api = HfApi(token=HF_TOKEN)

create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)

api.upload_folder(
    folder_path=CHECKPOINT_DIR,
    repo_id=HF_MODEL_REPO,
    repo_type='model',
    commit_message='Initial fine-tuned checkpoint — fintech-privacy-filter-v0',
)

model_url = f'https://huggingface.co/{HF_MODEL_REPO}'
wandb.log({'hf_model_url': model_url})
print(f'Model pushed: {model_url}')

In [ ]:
local_model = snapshot_download(repo_id=HF_MODEL_REPO, token=HF_TOKEN)
print('Model downloaded to:', local_model)

test_texts = [
    'Wire transfer to IBAN DE89370400440532013000 completed.',
    'The LEI 529900T8BM49AURSDO55 was submitted to the trade repository.',
    'Contact john.smith@acmecorp.com for invoice queries.',
]

table = wandb.Table(columns=['input', 'redacted'])

for text in test_texts:
    res = subprocess.run(
        ['opf', 'redact', '--checkpoint', local_model, text],
        capture_output=True, text=True
    )
    redacted = res.stdout.strip()
    print(f'Input:    {text}')
    print(f'Redacted: {redacted}\n')
    table.add_data(text, redacted)

wandb.log({'inference_examples': table})

In [ ]:
wandb.finish()
print('W&B run complete. View at:', run.url)